# FIT5201 Assignment 2 - Section 2: Perceptron vs Neural Networks
**Student Name**: [Yin Zhang]
**Student ID**: [[Student ID removed]]

## Question 1: Perceptron vs Neural Networks

### I. Data exploration (4 marks)

In [ ]:
# I. Data exploration (4 marks)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

%matplotlib inline

def save_show(path=None):
    """
    Save the current figure if a path is given, then show it in the notebook.
    This ensures the figure appears in exported HTML/PDF files.
    """
    if path:
        plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


# Load Task2B_train.csv and Task2B_test.csv
train_df = pd.read_csv('Dataset_S2_2025/Task2B_train.csv')
test_df = pd.read_csv('Dataset_S2_2025/Task2B_test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Testing data shape: {test_df.shape}")

print("\nTraining data:")
print(train_df.head())

print("\nTesting data:")
print(test_df.head())

In [ ]:
# Convert class labels from [0, 1] to [-1, +1]
y_train = train_df.iloc[:, -1].values
y_test = test_df.iloc[:, -1].values

X_train = train_df.iloc[:, :-1].values
X_test = test_df.iloc[:, :-1].values

y_train = np.where(y_train == 0, -1, 1)
y_test = np.where(y_test == 0, -1, 1)

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Plot training and testing data
plt.figure(figsize=(12, 5))

# Training data plot
plt.subplot(1, 2, 1)
plt.scatter(
    X_train[y_train == 1, 0],
    X_train[y_train == 1, 1],
    c='red',
    label='Class +1',
    alpha=0.6
)
plt.scatter(
    X_train[y_train == -1, 0],
    X_train[y_train == -1, 1],
    c='blue',
    label='Class -1',
    alpha=0.6
)
plt.title('Training Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

# Testing data plot
plt.subplot(1, 2, 2)
plt.scatter(
    X_test[y_test == 1, 0],
    X_test[y_test == 1, 1],
    c='red',
    label='Class +1',
    alpha=0.6
)
plt.scatter(
    X_test[y_test == -1, 0],
    X_test[y_test == -1, 1],
    c='blue',
    label='Class -1',
    alpha=0.6
)
plt.title('Testing Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
save_show('task2b_data_plots.png')


# Check whether the data is linearly separable using a linear SVM heuristic
def is_linearly_separable(X, y):
    """
    Check whether the data appears linearly separable using a high-C linear SVM.

    This is a practical heuristic, not a formal mathematical proof.
    If the linear SVM achieves 100% training accuracy, the data is treated as
    linearly separable for this exploratory analysis.
    """
    from sklearn.svm import SVC

    svm = SVC(kernel='linear', C=1e10)
    svm.fit(X, y)
    return svm.score(X, y) == 1.0


train_separable = is_linearly_separable(X_train, y_train)
test_separable = is_linearly_separable(X_test, y_test)

print(f"\nTraining data linearly separable: {train_separable}")
print(f"Testing data linearly separable: {test_separable}")

# Markdown discussion for the report
data_md = f"""
### I. Data exploration discussion

The training set contains **{X_train.shape[0]} samples** with **{X_train.shape[1]} input features**, while the test set contains **{X_test.shape[0]} samples** with the same two input features. The class labels were converted from **{{0, 1}}** to **{{-1, +1}}** because the Perceptron implementation uses labels in this form.

The scatter plots show that the two classes overlap in the two-dimensional feature space. The linear SVM heuristic gives:

- Training data linearly separable: **{train_separable}**
- Testing data linearly separable: **{test_separable}**

This suggests that the data is **not perfectly linearly separable**. Therefore, a single linear Perceptron may have limited performance because it can only learn a linear decision boundary. A neural network may be more suitable because it can represent nonlinear decision boundaries.
"""

display(Markdown(data_md))

### II. Perceptron models (8 marks)

In [ ]:
# L2-regularized Perceptron implementation
class RegularizedPerceptron:
    def __init__(self, learning_rate=0.01, lambda_reg=0.001, max_iter=1000,
                 early_stopping=False, patience=10, val_improvement_tol=0.001):
        self.learning_rate = learning_rate
        self.lambda_reg = lambda_reg
        self.max_iter = max_iter
        self.early_stopping = early_stopping
        self.patience = patience
        self.val_improvement_tol = val_improvement_tol
        self.weights = None
        self.bias = None
        self.train_errors = []

    def fit(self, X, y, X_val=None, y_val=None):
        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.train_errors = []

        best_weights = self.weights.copy()
        best_bias = self.bias
        best_val_error = float('inf')
        patience_counter = 0

        for epoch in range(self.max_iter):
            # Shuffle samples each epoch for Perceptron training
            indices = np.random.permutation(n_samples)

            for i in indices:
                prediction_score = np.dot(self.weights, X[i]) + self.bias

                # L2 weight decay; bias is not regularised
                self.weights *= (1 - self.learning_rate * self.lambda_reg)

                # Perceptron update if the sample is misclassified
                if prediction_score * y[i] <= 0:
                    self.weights += self.learning_rate * y[i] * X[i]
                    self.bias += self.learning_rate * y[i]

            train_error = np.mean(self.predict(X) != y)
            self.train_errors.append(train_error)

            if self.early_stopping and X_val is not None and y_val is not None:
                val_error = np.mean(self.predict(X_val) != y_val)

                if val_error < best_val_error - self.val_improvement_tol:
                    best_val_error = val_error
                    best_weights = self.weights.copy()
                    best_bias = self.bias
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= self.patience:
                    break

        if self.early_stopping and X_val is not None and y_val is not None:
            self.weights = best_weights
            self.bias = best_bias

        return self

    def predict(self, X):
        scores = np.dot(X, self.weights) + self.bias
        return np.where(scores >= 0, 1, -1)

    def decision_boundary(self, X):
        """
        Return the decision boundary line:
        w1*x1 + w2*x2 + b = 0
        """
        if self.weights is None:
            return None

        if abs(self.weights[1]) > 1e-10:
            x1 = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
            x2 = (-self.bias - self.weights[0] * x1) / self.weights[1]
            return x1, x2

        return None

In [ ]:
# Hyperparameter grid search for Perceptron

learning_rates = [0.001, 0.01, 0.1]
lambda_regs = [0.0001, 0.001, 0.01, 0.1, 1.0]

# Split training data into train and validation sets (80/20)
np.random.seed(42)
indices = np.random.permutation(len(X_train))
val_size = int(0.2 * len(X_train))

val_idx = indices[:val_size]
train_idx = indices[val_size:]

X_train_split = X_train[train_idx]
y_train_split = y_train[train_idx]
X_val = X_train[val_idx]
y_val = y_train[val_idx]

print(f"Training split shape: {X_train_split.shape}")
print(f"Validation split shape: {X_val.shape}")

best_perceptron_no_early_stop = None
best_perceptron_no_early_stop_error = float('inf')
best_params_no_early_stop = None

best_perceptron_with_early_stop = None
best_perceptron_with_early_stop_error = float('inf')
best_params_with_early_stop = None


print("Grid search for Perceptron without early stopping...")

for lr in learning_rates:
    for lam in lambda_regs:
        perceptron = RegularizedPerceptron(
            learning_rate=lr,
            lambda_reg=lam,
            max_iter=1000,
            early_stopping=False
        )

        perceptron.fit(
            X_train_split,
            y_train_split,
            X_val=X_val,
            y_val=y_val
        )

        val_pred = perceptron.predict(X_val)
        val_error = np.mean(val_pred != y_val)

        print(f"No early stopping - lr={lr}, lambda={lam}, val error={val_error:.4f}")

        if val_error < best_perceptron_no_early_stop_error:
            best_perceptron_no_early_stop_error = val_error
            best_perceptron_no_early_stop = perceptron
            best_params_no_early_stop = {
                'learning_rate': lr,
                'lambda_reg': lam
            }


print("\nGrid search for Perceptron with early stopping...")

for lr in learning_rates:
    for lam in lambda_regs:
        perceptron = RegularizedPerceptron(
            learning_rate=lr,
            lambda_reg=lam,
            max_iter=1000,
            early_stopping=True,
            patience=10,
            val_improvement_tol=0.001
        )

        perceptron.fit(
            X_train_split,
            y_train_split,
            X_val=X_val,
            y_val=y_val
        )

        val_pred = perceptron.predict(X_val)
        val_error = np.mean(val_pred != y_val)

        print(f"With early stopping - lr={lr}, lambda={lam}, val error={val_error:.4f}")

        if val_error < best_perceptron_with_early_stop_error:
            best_perceptron_with_early_stop_error = val_error
            best_perceptron_with_early_stop = perceptron
            best_params_with_early_stop = {
                'learning_rate': lr,
                'lambda_reg': lam
            }


print("\nBest Perceptron without early stopping:")
print(f"Validation error: {best_perceptron_no_early_stop_error:.4f}")
print(f"Best parameters: {best_params_no_early_stop}")

print("\nBest Perceptron with early stopping:")
print(f"Validation error: {best_perceptron_with_early_stop_error:.4f}")
print(f"Best parameters: {best_params_with_early_stop}")

In [ ]:
# Evaluate selected Perceptron models on the test set
from IPython.display import Markdown, display

test_pred_no_early = best_perceptron_no_early_stop.predict(X_test)
test_error_no_early = np.mean(test_pred_no_early != y_test)

test_pred_with_early = best_perceptron_with_early_stop.predict(X_test)
test_error_with_early = np.mean(test_pred_with_early != y_test)

print(f"\nTest error - Perceptron without early stopping: {test_error_no_early:.4f}")
print(f"Test error - Perceptron with early stopping: {test_error_with_early:.4f}")


# Plot decision boundaries
plt.figure(figsize=(14, 6))

# Perceptron without early stopping
plt.subplot(1, 2, 1)
plt.scatter(
    X_test[y_test == 1, 0],
    X_test[y_test == 1, 1],
    c='red',
    label='Class +1',
    alpha=0.6
)
plt.scatter(
    X_test[y_test == -1, 0],
    X_test[y_test == -1, 1],
    c='blue',
    label='Class -1',
    alpha=0.6
)

boundary = best_perceptron_no_early_stop.decision_boundary(X_test)
if boundary is not None:
    x1, x2 = boundary
    plt.plot(x1, x2, 'k-', linewidth=2, label='Decision Boundary')

plt.title('Perceptron without Early Stopping')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)


# Perceptron with early stopping
plt.subplot(1, 2, 2)
plt.scatter(
    X_test[y_test == 1, 0],
    X_test[y_test == 1, 1],
    c='red',
    label='Class +1',
    alpha=0.6
)
plt.scatter(
    X_test[y_test == -1, 0],
    X_test[y_test == -1, 1],
    c='blue',
    label='Class -1',
    alpha=0.6
)

boundary = best_perceptron_with_early_stop.decision_boundary(X_test)
if boundary is not None:
    x1, x2 = boundary
    plt.plot(x1, x2, 'k-', linewidth=2, label='Decision Boundary')

plt.title('Perceptron with Early Stopping')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
save_show('task2b_perceptron_decision_boundaries.png')


# Markdown discussion
delta_early = test_error_no_early - test_error_with_early

early_md = f"""### II. Perceptron test performance and early stopping

The selected Perceptron models were evaluated on the test set after hyperparameter selection using the validation set.

- **Without early stopping**
  - Best validation parameters: η = **{best_params_no_early_stop['learning_rate']}**, λ = **{best_params_no_early_stop['lambda_reg']}**
  - Test error = **{test_error_no_early:.4f}**

- **With early stopping**
  - Best validation parameters: η = **{best_params_with_early_stop['learning_rate']}**, λ = **{best_params_with_early_stop['lambda_reg']}**
  - Test error = **{test_error_with_early:.4f}**

The difference in test error is **{delta_early:+.4f}**, computed as:

**no-early-stopping test error − early-stopping test error**

Therefore, a positive value means that early stopping achieved a lower test error.
"""

if test_error_with_early < test_error_no_early:
    early_md += (
        "\n\nIn this run, early stopping improved test performance. "
        "The improvement is small, but it is consistent with the validation result and suggests slightly better generalisation. "
        "The decision boundary plots also show that both Perceptron models are limited by their linear decision boundary, "
        "which cannot fully capture the curved class structure in the data."
    )
elif test_error_with_early > test_error_no_early:
    early_md += (
        "\n\nIn this run, early stopping increased test error. "
        "This can happen if the validation split is unlucky or if training stops before the model reaches a better decision boundary. "
        "The decision boundary plots still show that both Perceptron models are limited by their linear decision boundary, "
        "which cannot fully capture the curved class structure in the data."
    )
else:
    early_md += (
        "\n\nBoth settings achieved the same test error on this split. "
        "The decision boundary plots show that both Perceptron models are limited by their linear decision boundary, "
        "which cannot fully capture the curved class structure in the data."
    )

display(Markdown(early_md))

### III. Neural network models (8 marks)

In [ ]:
# 3-layer Neural Network implementation
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, lambda_reg=0.001, learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.lambda_reg = lambda_reg
        self.learning_rate = learning_rate

        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros(output_size)

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def sigmoid_derivative(self, z):
        s = self.sigmoid(z)
        return s * (1 - s)

    def _targets(self, y):
        """Map labels {-1, +1} to sigmoid targets {0, 1}."""
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        if np.any(y < 0):
            y = (y + 1) / 2.0
        return y

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.sigmoid(self.Z1)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.sigmoid(self.Z2)
        return self.A2

    def backward(self, X, y, output):
        m = X.shape[0]
        y = self._targets(y)

        dZ2 = output - y
        dW2 = (1 / m) * np.dot(self.A1.T, dZ2) + (self.lambda_reg / m) * self.W2
        db2 = (1 / m) * np.sum(dZ2, axis=0)

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.sigmoid_derivative(self.Z1)
        dW1 = (1 / m) * np.dot(X.T, dZ1) + (self.lambda_reg / m) * self.W1
        db1 = (1 / m) * np.sum(dZ1, axis=0)

        return dW1, db1, dW2, db2

    def fit(self, X, y, epochs=1000, batch_size=32, early_stopping=False, patience=10, X_val=None, y_val=None):
        m = X.shape[0]

        best_W1, best_b1 = None, None
        best_W2, best_b2 = None, None
        best_error = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            indices = np.random.permutation(m)

            for i in range(0, m, batch_size):
                batch_idx = indices[i:i + batch_size]
                xb = X[batch_idx]
                yb = y[batch_idx]

                output = self.forward(xb)
                dW1, db1, dW2, db2 = self.backward(xb, yb, output)

                self.W1 -= self.learning_rate * dW1
                self.b1 -= self.learning_rate * db1
                self.W2 -= self.learning_rate * dW2
                self.b2 -= self.learning_rate * db2

            if early_stopping and X_val is not None and y_val is not None:
                val_error = np.mean(self.predict(X_val) != y_val)

                if val_error < best_error:
                    best_error = val_error
                    best_W1 = self.W1.copy()
                    best_b1 = self.b1.copy()
                    best_W2 = self.W2.copy()
                    best_b2 = self.b2.copy()
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    break

        if early_stopping and best_W1 is not None:
            self.W1 = best_W1
            self.b1 = best_b1
            self.W2 = best_W2
            self.b2 = best_b2

        return self

    def predict(self, X):
        output = self.forward(X).ravel()
        return np.where(output >= 0.5, 1, -1)

In [ ]:
# Hyperparameter grid search for Neural Networks

hidden_sizes = [5, 10, 15, 20, 25, 30, 35, 40]
learning_rates = [0.001, 0.01, 0.1]
lambda_regs = [0.001, 1.0]

best_nn_lambda_0_001 = None
best_nn_lambda_0_001_error = float('inf')
best_params_nn_lambda_0_001 = None

best_nn_lambda_1_0 = None
best_nn_lambda_1_0_error = float('inf')
best_params_nn_lambda_1_0 = None


print("Grid search for Neural Network with lambda=0.001...")

for hidden_size in hidden_sizes:
    for lr in learning_rates:
        nn = NeuralNetwork(
            input_size=2,
            hidden_size=hidden_size,
            output_size=1,
            lambda_reg=0.001,
            learning_rate=lr
        )

        nn.fit(
            X_train_split,
            y_train_split,
            epochs=1000,
            batch_size=32,
            early_stopping=True,
            patience=10,
            X_val=X_val,
            y_val=y_val
        )

        val_pred = nn.predict(X_val)
        val_error = np.mean(val_pred != y_val)

        print(f"lambda=0.001, hidden={hidden_size}, lr={lr}, val error={val_error:.4f}")

        if val_error < best_nn_lambda_0_001_error:
            best_nn_lambda_0_001_error = val_error
            best_nn_lambda_0_001 = nn
            best_params_nn_lambda_0_001 = {
                "hidden_size": hidden_size,
                "learning_rate": lr,
                "lambda_reg": 0.001
            }


print("\nGrid search for Neural Network with lambda=1.0...")

for hidden_size in hidden_sizes:
    for lr in learning_rates:
        nn = NeuralNetwork(
            input_size=2,
            hidden_size=hidden_size,
            output_size=1,
            lambda_reg=1.0,
            learning_rate=lr
        )

        nn.fit(
            X_train_split,
            y_train_split,
            epochs=1000,
            batch_size=32,
            early_stopping=True,
            patience=10,
            X_val=X_val,
            y_val=y_val
        )

        val_pred = nn.predict(X_val)
        val_error = np.mean(val_pred != y_val)

        print(f"lambda=1.0, hidden={hidden_size}, lr={lr}, val error={val_error:.4f}")

        if val_error < best_nn_lambda_1_0_error:
            best_nn_lambda_1_0_error = val_error
            best_nn_lambda_1_0 = nn
            best_params_nn_lambda_1_0 = {
                "hidden_size": hidden_size,
                "learning_rate": lr,
                "lambda_reg": 1.0
            }


print("\nBest Neural Network with lambda=0.001:")
print(f"Validation error: {best_nn_lambda_0_001_error:.4f}")
print(f"Best parameters: {best_params_nn_lambda_0_001}")

print("\nBest Neural Network with lambda=1.0:")
print(f"Validation error: {best_nn_lambda_1_0_error:.4f}")
print(f"Best parameters: {best_params_nn_lambda_1_0}")

In [ ]:
# Evaluate selected Neural Network models on the test set with 3 random initialisations
from IPython.display import Markdown, display

def train_and_test_nn(params, X_train_split, y_train_split, X_val, y_val, X_test, y_test, seed):
    """
    Train one neural network with selected hyperparameters and return test error.
    """
    np.random.seed(seed)

    nn = NeuralNetwork(
        input_size=2,
        hidden_size=params["hidden_size"],
        output_size=1,
        lambda_reg=params["lambda_reg"],
        learning_rate=params["learning_rate"]
    )

    nn.fit(
        X_train_split,
        y_train_split,
        epochs=1000,
        batch_size=32,
        early_stopping=True,
        patience=10,
        X_val=X_val,
        y_val=y_val
    )

    test_pred = nn.predict(X_test)
    test_error = np.mean(test_pred != y_test)

    return nn, test_error


def evaluate_nn_with_initializations(params, n_runs=3):
    """
    Retrain the selected NN setting with 3 random initialisations.
    """
    models = []
    errors = []

    for i in range(n_runs):
        model, error = train_and_test_nn(
            params,
            X_train_split,
            y_train_split,
            X_val,
            y_val,
            X_test,
            y_test,
            seed=100 + i
        )
        models.append(model)
        errors.append(error)

    best_run_idx = int(np.argmin(errors))
    return models[best_run_idx], np.mean(errors), np.std(errors), errors


best_nn_model_0_001, test_error_nn_0_001, test_std_nn_0_001, errors_0_001 = evaluate_nn_with_initializations(
    best_params_nn_lambda_0_001,
    n_runs=3
)

best_nn_model_1_0, test_error_nn_1_0, test_std_nn_1_0, errors_1_0 = evaluate_nn_with_initializations(
    best_params_nn_lambda_1_0,
    n_runs=3
)

print(f"\nNeural Network (lambda=0.001) test errors over 3 runs: {errors_0_001}")
print(f"Neural Network (lambda=0.001) - Test error: {test_error_nn_0_001:.4f} ± {test_std_nn_0_001:.4f}")

print(f"\nNeural Network (lambda=1.0) test errors over 3 runs: {errors_1_0}")
print(f"Neural Network (lambda=1.0) - Test error: {test_error_nn_1_0:.4f} ± {test_std_nn_1_0:.4f}")

In [ ]:
def plot_nn_decision_boundary(model, X, y, title, filename):
    """
    Plot the nonlinear decision boundary of a neural network.
    The boundary is where model output = 0.5.
    """
    x_min, x_max = X[:, 0].min() - 0.2, X[:, 0].max() + 0.2
    y_min, y_max = X[:, 1].min() - 0.2, X[:, 1].max() + 0.2

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = model.forward(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 6))

    plt.contourf(xx, yy, probs, levels=[0, 0.5, 1], alpha=0.25)
    plt.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)

    plt.scatter(
        X[y == 1, 0],
        X[y == 1, 1],
        c='red',
        label='Class +1',
        alpha=0.6
    )
    plt.scatter(
        X[y == -1, 0],
        X[y == -1, 1],
        c='blue',
        label='Class -1',
        alpha=0.6
    )

    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()
    plt.grid(True, alpha=0.3)

    save_show(filename)


plot_nn_decision_boundary(
    best_nn_model_0_001,
    X_test,
    y_test,
    f"Neural Network Decision Boundary (λ=0.001)\nTest Error: {test_error_nn_0_001:.4f} ± {test_std_nn_0_001:.4f}",
    "task2b_nn_lambda_0001_decision_boundary.png"
)

plot_nn_decision_boundary(
    best_nn_model_1_0,
    X_test,
    y_test,
    f"Neural Network Decision Boundary (λ=1.0)\nTest Error: {test_error_nn_1_0:.4f} ± {test_std_nn_1_0:.4f}",
    "task2b_nn_lambda_1_0_decision_boundary.png"
)

nn_eval_md = f"""
### III. Neural Network test performance

After selecting hyperparameters using the validation set, each selected neural network setting was retrained with **3 random initialisations**.

- **NN with λ = 0.001**
  - Selected hidden size: **{best_params_nn_lambda_0_001['hidden_size']}**
  - Selected learning rate: **{best_params_nn_lambda_0_001['learning_rate']}**
  - Validation error: **{best_nn_lambda_0_001_error:.4f}**
  - Test error over 3 runs: **{test_error_nn_0_001:.4f} ± {test_std_nn_0_001:.4f}**

- **NN with λ = 1.0**
  - Selected hidden size: **{best_params_nn_lambda_1_0['hidden_size']}**
  - Selected learning rate: **{best_params_nn_lambda_1_0['learning_rate']}**
  - Validation error: **{best_nn_lambda_1_0_error:.4f}**
  - Test error over 3 runs: **{test_error_nn_1_0:.4f} ± {test_std_nn_1_0:.4f}**

The decision boundary plots show that the selected neural networks learn relatively smooth and almost linear boundaries in this run. 
Although neural networks can represent nonlinear decision boundaries in principle, the selected settings do not clearly outperform the Perceptron on the test set. 
The λ = 1.0 model has a slightly lower mean test error than the λ = 0.001 model, 
suggesting that stronger regularisation gives slightly more stable generalisation here..
"""

display(Markdown(nn_eval_md))

### IV. Comparative analysis (5 marks)

In [ ]:
from IPython.display import Markdown, display

model_errors = {
    'Perceptron (no early stopping)': test_error_no_early,
    'Perceptron (with early stopping)': test_error_with_early,
    'Neural Network (λ=0.001)': test_error_nn_0_001,
    'Neural Network (λ=1.0)': test_error_nn_1_0,
}

best_model = min(model_errors, key=model_errors.get)
best_err = model_errors[best_model]

linear_best = min(test_error_no_early, test_error_with_early)
nn_best = min(test_error_nn_0_001, test_error_nn_1_0)

print('\n=== Test errors (same as Markdown below) ===')
for k, v in model_errors.items():
    print(f'  {k}: {v:.4f}')

cmp_md = f"""### IV. Comparative analysis discussion

**Test errors**

| Model | Test error |
|-------|------------|
| Perceptron without early stopping | {test_error_no_early:.4f} |
| Perceptron with early stopping | {test_error_with_early:.4f} |
| NN λ = 0.001 | {test_error_nn_0_001:.4f} ± {test_std_nn_0_001:.4f} |
| NN λ = 1.0 | {test_error_nn_1_0:.4f} ± {test_std_nn_1_0:.4f} |

1. **Best model on the test set:** **{best_model}**, with test error **{best_err:.4f}**.
"""

if nn_best < linear_best:
    cmp_md += (
        f"\nThe lowest NN error ({nn_best:.4f}) is lower than the best Perceptron error ({linear_best:.4f}). "
        "This suggests that the neural network benefits from its greater representational capacity on this dataset."
    )
elif nn_best > linear_best:
    cmp_md += (
        f"\nThe best Perceptron error ({linear_best:.4f}) is lower than the best NN error ({nn_best:.4f}) in this run. "
        "Although the data is not perfectly linearly separable, the selected NN settings do not clearly outperform the early-stopping Perceptron."
    )
else:
    cmp_md += (
        "\nThe best Perceptron and best NN achieve the same test error in this run."
    )

cmp_md += "\n\n2. **Underfitting / overfitting signal:**\n"

if test_error_with_early < test_error_no_early:
    cmp_md += (
        f"- Early stopping reduced the Perceptron test error from **{test_error_no_early:.4f}** "
        f"to **{test_error_with_early:.4f}**, suggesting slightly better generalisation.\n"
    )
elif test_error_with_early > test_error_no_early:
    cmp_md += (
        f"- Early stopping increased the Perceptron test error from **{test_error_no_early:.4f}** "
        f"to **{test_error_with_early:.4f}**, suggesting possible under-training or validation-split sensitivity.\n"
    )
else:
    cmp_md += (
        f"- Early stopping produced the same Perceptron test error (**{test_error_with_early:.4f}**) as no early stopping.\n"
    )

if test_error_nn_1_0 > test_error_nn_0_001:
    cmp_md += (
        f"- Stronger regularisation with λ = 1.0 increased the NN test error "
        f"({test_error_nn_1_0:.4f} vs {test_error_nn_0_001:.4f}), suggesting possible underfitting.\n"
    )
elif test_error_nn_1_0 < test_error_nn_0_001:
    cmp_md += (
        f"- Stronger regularisation with λ = 1.0 achieved lower NN test error "
        f"({test_error_nn_1_0:.4f} vs {test_error_nn_0_001:.4f}), suggesting more stable generalisation here.\n"
    )
else:
    cmp_md += (
        "- Both NN regularisation settings achieved the same mean test error across initialisations.\n"
    )

cmp_md += f"""
3. **Decision boundaries:** The Perceptron is restricted to a single linear decision boundary. A 3-layer NN can represent nonlinear boundaries in principle, but the selected NN models in this run learn relatively smooth and almost linear boundaries. This helps explain why the NN models do not clearly outperform the early-stopping Perceptron.

4. **Regularisation:** A larger λ penalises large weights more strongly. Here, λ = 1.0 gives a slightly lower mean NN test error than λ = 0.001, indicating that stronger regularisation is helpful for this split.

5. **Random initialisation:** The NN models were retrained with 3 random initialisations. The standard deviations are **{test_std_nn_0_001:.4f}** for λ = 0.001 and **{test_std_nn_1_0:.4f}** for λ = 1.0. A larger standard deviation indicates greater sensitivity to initial weights.
"""

display(Markdown(cmp_md))